In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
path_root = Path.cwd()

mlflow.set_tracking_uri(f"file:{path_root / 'mlruns'}")
experiment = mlflow.set_experiment("stock-lstm-AAPL")

print(f"Project root:  {path_root}")
print(f"Tracking URI:  {mlflow.get_tracking_uri()}")
print(f"Experiment:    {experiment.name}")

Project root:  /home/caio/projetos/tech-challenge-fase4-lstm
Tracking URI:  file:/home/caio/projetos/tech-challenge-fase4-lstm/mlruns
Experiment:    stock-lstm-AAPL


In [2]:

close_df = pd.read_csv(path_root / "reports" / "AAPL" / "close_prices.csv")
print(f"Total de linhas: {len(close_df)}")
print(f"Colunas: {close_df.columns.tolist()}")
print(close_df.head())
print(close_df.tail())

with open(path_root / "models" / "AAPL" / "metadata.json") as f:
    metadata = json.load(f)

sequence_length = metadata["sequence_length"]
train_samples = metadata["train_samples"]


reference_start = sequence_length
reference_end = sequence_length + train_samples
production_start = reference_end

reference_df = close_df.iloc[reference_start:reference_end].reset_index(drop=True)
production_df = close_df.iloc[production_start:].reset_index(drop=True)

print(f"\nReference (período treino do modelo): {len(reference_df)} linhas")
print(f"Production (período test do modelo):  {len(production_df)} linhas")
print(f"\nReference - primeiras datas:\n{reference_df.head(3)}")
print(f"\nReference - últimas datas:\n{reference_df.tail(3)}")
print(f"\nProduction - primeiras datas:\n{production_df.head(3)}")
print(f"\nProduction - últimas datas:\n{production_df.tail(3)}")

Total de linhas: 1647
Colunas: ['Date', 'Close']
         Date      Close
0  2018-01-02  43.064999
1  2018-01-03  43.057499
2  2018-01-04  43.257500
3  2018-01-05  43.750000
4  2018-01-08  43.587502
            Date       Close
1642  2024-07-15  234.399994
1643  2024-07-16  234.820007
1644  2024-07-17  228.880005
1645  2024-07-18  224.179993
1646  2024-07-19  224.309998

Reference (período treino do modelo): 1269 linhas
Production (período test do modelo):  318 linhas

Reference - primeiras datas:
         Date      Close
0  2018-03-29  41.945000
1  2018-04-02  41.669998
2  2018-04-03  42.097500

Reference - últimas datas:
            Date       Close
1266  2023-04-11  160.800003
1267  2023-04-12  160.100006
1268  2023-04-13  165.559998

Production - primeiras datas:
         Date       Close
0  2023-04-14  165.210007
1  2023-04-17  165.229996
2  2023-04-18  166.470001

Production - últimas datas:
           Date       Close
315  2024-07-17  228.880005
316  2024-07-18  224.179993
317  

In [3]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

reference_data = reference_df[["Close"]].copy()
production_data = production_df[["Close"]].copy()

print(f"Reference shape: {reference_data.shape}")
print(f"Production shape: {production_data.shape}")
print(f"\nReference stats:\n{reference_data.describe()}")
print(f"\nProduction stats:\n{production_data.describe()}")

drift_report = Report(metrics=[
    DataDriftPreset(),
    DataQualityPreset(),
])

drift_report.run(reference_data=reference_data, current_data=production_data)

reports_dir = path_root / "mlops" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
html_path = reports_dir / "evidently_data_drift_AAPL.html"
drift_report.save_html(str(html_path))

print(f"\n✅ Report salvo em: {html_path}")
print(f"   Tamanho: {html_path.stat().st_size / 1024:.1f} KB")

drift_dict = drift_report.as_dict()
drift_detected = drift_dict["metrics"][0]["result"]["dataset_drift"]
print(f"\nDrift detectado no dataset: {drift_detected}")

Reference shape: (1269, 1)
Production shape: (318, 1)

Reference stats:
             Close
count  1269.000000
mean    103.523928
std      45.891991
min      35.547501
25%      53.872501
50%     115.970001
75%     146.100006
max     182.009995

Production stats:
            Close
count  318.000000
mean   184.590252
std     13.998236
min    163.759995
25%    173.794998
50%    182.709999
75%    191.277496
max    234.820007

✅ Report salvo em: /home/caio/projetos/tech-challenge-fase4-lstm/mlops/reports/evidently_data_drift_AAPL.html
   Tamanho: 2942.8 KB

Drift detectado no dataset: True


In [ ]:
from IPython.display import IFrame

with mlflow.start_run(run_name="drift_detection_reference_vs_production") as run:
    mlflow.set_tag("ticker", "AAPL")
    mlflow.set_tag("analysis_type", "data_drift")
    mlflow.set_tag("tool", "evidently")
    mlflow.set_tag("reference_period", f"{reference_df['Date'].iloc[0]} to {reference_df['Date'].iloc[-1]}")
    mlflow.set_tag("production_period", f"{production_df['Date'].iloc[0]} to {production_df['Date'].iloc[-1]}")

    mlflow.log_metric("reference_size", len(reference_data))
    mlflow.log_metric("production_size", len(production_data))
    mlflow.log_metric("reference_close_mean", float(reference_data["Close"].mean()))
    mlflow.log_metric("production_close_mean", float(production_data["Close"].mean()))
    mlflow.log_metric("reference_close_std", float(reference_data["Close"].std()))
    mlflow.log_metric("production_close_std", float(production_data["Close"].std()))
    mlflow.log_metric("drift_detected", int(drift_detected))

    mlflow.log_artifact(str(html_path), artifact_path="drift_reports")

    print(f" Run de drift logado: {run.info.run_id}")
    print(f" Drift detectado: {bool(drift_detected)}")

relative_path = f"../reports/{html_path.name}"
print(f"\n Report disponível em: {html_path}")
IFrame(src=relative_path, width="100%", height=800)

✅ Run de drift logado: 061b30820a1b442f95a2db3020f65fbe
   Drift detectado: True
   HTML logado em: drift_reports/evidently_data_drift_AAPL.html

📊 Abrindo report inline (também disponível em /home/caio/projetos/tech-challenge-fase4-lstm/mlops/reports/evidently_data_drift_AAPL.html):


ValueError: '/home/caio/projetos/tech-challenge-fase4-lstm/mlops/reports/evidently_data_drift_AAPL.html' is not in the subpath of '/home/caio/projetos/tech-challenge-fase4-lstm/mlops/notebooks' OR one path is relative and the other is absolute.